# Tools v0.2：给组员看的最小接入例子

本 Notebook 默认使用**完全虚构数据**，不联网、不调用 LLM。接口与真实 Yahoo 模式一致。

我们已经从“一个 Python 函数”走到了“统一的工具入口”：`工具名 + 参数字典 → 结果字典`。工具输入规格见 `tool_specs.json`，完整说明见 `TOOLS_HANDOFF.md`。

In [ ]:
import sys
from pathlib import Path
import json

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "equity_tools.py").exists():
    PROJECT_DIR = PROJECT_DIR / "elec7082-equity-agent"
if not (PROJECT_DIR / "equity_tools.py").exists():
    raise RuntimeError("请从项目文件夹打开此 Notebook。")
sys.path.insert(0, str(PROJECT_DIR.resolve()))
from equity_tools import ToolClient
from demo_tools_provider import DemoTicker

client = ToolClient(output_dir=PROJECT_DIR / "data" / "tools_notebook_demo",
                    ticker_factory=DemoTicker, demo=True)
print("当前模式：完全虚构的教学数据")

## 1. 调用行情工具

`arguments` 是普通 Python 字典。日线区间包括开始日期，不包括结束日期。

In [ ]:
arguments = {"symbol": "AAPL", "start": "2024-01-02", "end": "2024-01-04"}
result = client.call_tool("get_price_history", arguments)
print(result["status"])
print(json.dumps(result["data"], ensure_ascii=False, indent=2))
print(result["warnings"])

## 2. 调用公司概况工具

现在换一个工具名，入口和返回结构都不变。这里的公司名字会明确显示为 SYNTHETIC DEMO。

In [ ]:
profile = client.call_tool("get_company_profile", {"symbol": "AAPL"})
print(profile["data"]["company_name"])
print(profile["data"]["quote_currency"])
print(profile["data"]["financial_currency"])

## 3. 调用财报工具

`annual` 是年度，`quarterly` 是季度。示例从新到旧返回两个期间。金额、单位、原始行列和来源 ID 分开保存。

In [ ]:
financials = client.call_tool("get_financial_statements", {
    "symbol": "AAPL", "frequency": "annual", "periods": 2,
})
print(financials["status"])
income = financials["data"]["statements"]["income_statement"]["periods"][0]
print("期末：", income["period_end"])
print("收入：", income["metrics"]["revenue"])
print("来源 ID：", income["source_id"])

## 4. 找回证据文件

来源路径相对于 `client.output_dir`。哈希只证明保存内容未改变，不能证明供应商每个数字都正确。

In [ ]:
import hashlib
source = financials["sources"][-1]
raw_file = client.output_dir / source["raw_path"]
print("证据文件：", raw_file)
assert hashlib.sha256(raw_file.read_bytes()).hexdigest() == source["sha256"]
print("文件 SHA-256 核对通过")

## 5. 看看非法请求怎样返回

未知公司不会发出数据请求，返回 `INVALID_ARGUMENTS`，而不是虚构股票数据。

In [ ]:
bad_request = client.call_tool("get_company_profile", {"symbol": "INVALID"})
print(json.dumps(bad_request, ensure_ascii=False, indent=2))

## 6. 准备好后再运行真实模式

下面是**可选的联网代码**，默认全部注释。它访问 Yahoo，不需要 LLM API Key。你们选择的三家公司共用同一接口。

真实模式会返回当前供应商版本，不能直接用于历史 as-of 回测；原始 SEC 财报和电话会文字稿是下一批工具。

In [ ]:
# live_client = ToolClient(output_dir=PROJECT_DIR / "data" / "tools_notebook_live")
# live_result = live_client.call_tool("get_financial_statements", {
#     "symbol": "AAPL", "frequency": "annual", "periods": 4,
# })
# print(json.dumps(live_result, ensure_ascii=False, indent=2))